# 선택적 API 재수집 도구
기본 분석은 `data/01_raw/team_route_results/`의 기존 수집 완료 데이터를 사용합니다.
`run_pipeline.py`는 이 노트북을 실행하지 않습니다. 재수집은 기존 결과와 다른 교통시간을 반환할 수 있습니다.
재수집이 필요한 경우에만 다음 셀의 `ENABLE_API_COLLECTION`을 명시적으로 변경하세요.


In [ ]:
ENABLE_API_COLLECTION = False
if not ENABLE_API_COLLECTION:
    raise RuntimeError("기본 분석은 저장된 API 데이터를 사용합니다. 재수집을 의도한 경우에만 ENABLE_API_COLLECTION=True로 변경하세요.")


# 04-2. 네이버 Directions 5 통학경로 분담 수집 — Google Colab

이 노트북은 `commute_route_requests.csv`에서 본인에게 지정된 배치만 골라
네이버 Directions 5로 도로거리와 도로시간을 수집합니다.

핵심 안전장치
- Google Drive에 결과를 누적 저장하므로 Colab 런타임이 끊겨도 재실행 가능
- 수집 셀을 실행할 때마다 최신 체크포인트를 다시 읽어 완료된 요청은 실제로 건너뜀
- 진행률은 전체 배치 기준으로 표시하므로 재실행 시 기존 완료 지점부터 보임
- 개별 API·좌표·응답 오류는 `API_ERROR`로 기록하고 전체 수집은 계속 진행
- 호출 속도 제한과 일시적 서버 오류는 지수 백오프로 재시도
- 계정별 최대 호출 수를 넘기기 전에는 비용·할당량 보호를 위해서만 중단
- API Key는 출력 파일과 로그에 저장하지 않음

주의
- API 오류 한 건 때문에 배치 전체가 중단되지는 않습니다.
- 다만 Colab 런타임 종료·사용자 중단·Drive 저장 실패·계정별 안전 호출 한도 도달은
  코드가 계속 실행할 수 없는 상황이므로 중단될 수 있습니다.
- 각 팀원은 네이버클라우드 약관상 사용할 권한이 있는 본인 계정과 API 키를 사용해야 합니다.

In [ ]:
from __future__ import annotations

from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm


## 1. 팀원별 설정

팀원 1은 `MY_BATCH_NUMBER = 1`, 팀원 2는 `MY_BATCH_NUMBER = 2`로 설정합니다.
Client ID와 Client Secret은 따옴표 안에 직접 입력합니다.
이 노트북을 다른 사람에게 다시 보낼 때는 반드시 키를 지우세요.


In [ ]:
# ===================== 반드시 수정할 값 =====================
TEAM_MEMBER_LABEL = "member_1"       # 예: member_1, member_2
MY_BATCH_NUMBER = 1                   # 팀원 1: 1 / 팀원 2: 2
TOTAL_BATCH_COUNT = 2

NAVER_MAPS_CLIENT_ID = "여기에_CLIENT_ID_입력"
NAVER_MAPS_CLIENT_SECRET = "여기에_CLIENT_SECRET_입력"
# ===========================================================

# Google Drive 폴더 구조
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/small_school_project")
INPUT_PATH = DRIVE_PROJECT_DIR / "input" / "commute_route_requests.csv"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "routes"

# API 및 수집 설정
API_URL = "https://maps.apigw.ntruss.com/map-direction/v1/driving"
ROUTE_OPTION = "traoptimal"
REQUESTS_PER_SECOND = 3.5       # 콘솔 설정 4회/초보다 여유 있게 설정
MAX_API_CALLS_PER_ACCOUNT = 59_000
MAX_RETRIES_PER_REQUEST = 4
CHECKPOINT_EVERY = 10           # 최대 10개 결과마다 Drive에 추가 저장
CONNECT_TIMEOUT_SEC = 10
READ_TIMEOUT_SEC = 30

KST = timezone(timedelta(hours=9))
TERMINAL_STATUSES = {"SUCCESS", "NO_ROUTE", "API_ERROR"}

if MY_BATCH_NUMBER < 1 or MY_BATCH_NUMBER > TOTAL_BATCH_COUNT:
    raise ValueError("MY_BATCH_NUMBER는 1부터 TOTAL_BATCH_COUNT 사이여야 합니다.")
if NAVER_MAPS_CLIENT_ID.startswith("여기에_") or NAVER_MAPS_CLIENT_SECRET.startswith("여기에_"):
    raise ValueError("NAVER_MAPS_CLIENT_ID와 NAVER_MAPS_CLIENT_SECRET을 직접 입력하세요.")
if REQUESTS_PER_SECOND <= 0:
    raise ValueError("REQUESTS_PER_SECOND는 0보다 커야 합니다.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_ID = f"B{MY_BATCH_NUMBER:02d}_of_{TOTAL_BATCH_COUNT:02d}"
RESULT_PATH = OUTPUT_DIR / f"naver_commute_routes_{BATCH_ID}.csv"
SUMMARY_PATH = OUTPUT_DIR / f"naver_commute_routes_{BATCH_ID}_summary.csv"

print(f"담당 배치: {BATCH_ID}")
print(f"입력 파일: {INPUT_PATH}")
print(f"결과 파일: {RESULT_PATH}")

## 2. 입력 파일 및 기존 체크포인트 확인


In [ ]:
REQUIRED_INPUT_COLUMNS = [
    "route_request_id",
    "census_id",
    "origin_lon",
    "origin_lat",
    "destination_school_key",
    "destination_school_name",
    "destination_school_lon",
    "destination_school_lat",
    "straight_distance_km",
    "request_batch_number",
    "request_batch_id",
]

RESULT_COLUMNS = [
    "route_request_id",
    "request_batch_number",
    "request_batch_id",
    "collector_label",
    "source_request_file_sha256",
    "route_option",
    "route_status",
    "route_check",
    "review_flag",
    "http_status",
    "api_code",
    "api_message",
    "road_distance_m",
    "road_duration_ms",
    "road_duration_sec",
    "road_duration_min",
    "road_to_straight_ratio",
    "api_current_datetime",
    "api_departure_time",
    "requested_at_kst",
    "collected_at_utc",
    "snapped_start_lon",
    "snapped_start_lat",
    "snapped_goal_lon",
    "snapped_goal_lat",
    "start_snap_distance_m",
    "goal_snap_distance_m",
    "ferry_flag",
    "attempt_count",
    "last_error",
]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_text(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, low_memory=False)


if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"입력 파일이 없습니다: {INPUT_PATH}\n"
        "Google Drive의 small_school_project/input 폴더에 "
        "commute_route_requests.csv를 넣으세요."
    )

source_hash = sha256_file(INPUT_PATH)
requests_all = read_csv_text(INPUT_PATH)
missing_columns = [column for column in REQUIRED_INPUT_COLUMNS if column not in requests_all.columns]
if missing_columns:
    raise ValueError(f"입력 파일 필수 칼럼 누락: {missing_columns}")
if requests_all["route_request_id"].duplicated().any():
    raise ValueError("입력 파일의 route_request_id가 중복되어 있습니다.")

for column in [
    "origin_lon",
    "origin_lat",
    "destination_school_lon",
    "destination_school_lat",
    "straight_distance_km",
    "request_batch_number",
]:
    requests_all[column] = pd.to_numeric(requests_all[column], errors="coerce")

if requests_all[REQUIRED_INPUT_COLUMNS].isna().any().any():
    null_counts = requests_all[REQUIRED_INPUT_COLUMNS].isna().sum()
    raise ValueError(f"입력 파일 필수값 결측:\n{null_counts[null_counts > 0]}")

batch_requests = requests_all[
    requests_all["request_batch_number"].eq(MY_BATCH_NUMBER)
].copy()
# 같은 집계구의 현재학교·통합학교 경로를 수초 이내에 연속 수집하되,
# 집계구 코드의 지역 순서가 수집 시각과 겹치지 않도록 집계구 순서는 해시로 섞는다.
batch_requests["_census_collection_order"] = batch_requests["census_id"].map(
    lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
)
batch_requests = batch_requests.sort_values(
    ["_census_collection_order", "census_id", "destination_school_key"],
    kind="stable",
).drop(columns="_census_collection_order").reset_index(drop=True)

expected_batch_id = f"B{MY_BATCH_NUMBER:02d}_of_{TOTAL_BATCH_COUNT:02d}"
if not batch_requests["request_batch_id"].eq(expected_batch_id).all():
    raise ValueError(
        "입력 파일의 request_batch_id와 노트북 설정이 다릅니다. "
        "먼저 04-1 코드를 --batch-count 2로 다시 실행했는지 확인하세요."
    )
if batch_requests.empty:
    raise ValueError(f"{BATCH_ID}에 해당하는 경로가 없습니다.")
if len(batch_requests) > MAX_API_CALLS_PER_ACCOUNT:
    raise ValueError(
        f"담당 경로 {len(batch_requests):,}건이 계정별 안전 한도 "
        f"{MAX_API_CALLS_PER_ACCOUNT:,}건보다 많습니다."
    )

if RESULT_PATH.exists() and RESULT_PATH.stat().st_size > 0:
    existing_results = read_csv_text(RESULT_PATH)
    missing_result_columns = [column for column in RESULT_COLUMNS if column not in existing_results.columns]
    if missing_result_columns:
        raise ValueError(
            f"기존 체크포인트 파일 칼럼이 현재 코드와 다릅니다: {missing_result_columns}"
        )
    stored_hashes = set(existing_results["source_request_file_sha256"].dropna())
    if stored_hashes and stored_hashes != {source_hash}:
        raise ValueError(
            "기존 체크포인트가 다른 commute_route_requests.csv에서 생성되었습니다. "
            "기존 결과를 별도 보관한 뒤 새로 시작하세요."
        )
else:
    existing_results = pd.DataFrame(columns=RESULT_COLUMNS)

existing_results["attempt_count_num"] = pd.to_numeric(
    existing_results.get("attempt_count", pd.Series(dtype=str)), errors="coerce"
).fillna(0)
api_calls_recorded = int(existing_results["attempt_count_num"].sum())

terminal_existing = existing_results[
    existing_results["route_status"].isin(TERMINAL_STATUSES)
]
done_ids = set(terminal_existing["route_request_id"])
pending_requests = batch_requests[~batch_requests["route_request_id"].isin(done_ids)].copy()

unexpected_existing_ids = set(existing_results["route_request_id"].dropna()) - set(
    batch_requests["route_request_id"]
)
if unexpected_existing_ids:
    raise ValueError(
        f"기존 결과에 다른 배치의 route_request_id가 포함되어 있습니다: "
        f"{list(sorted(unexpected_existing_ids))[:5]}"
    )

print(f"입력 파일 SHA-256: {source_hash}")
print(f"담당 경로: {len(batch_requests):,}건")
print(f"기존 완료·오류 기록으로 자동 건너뜀: {len(done_ids):,}건")
print(f"이번 실행 대기: {len(pending_requests):,}건")
print("※ 실제 수집 셀은 실행 직전에 체크포인트를 다시 읽어 이 수치를 갱신합니다.")
print(f"체크포인트에 기록된 API 호출 시도: {api_calls_recorded:,}회")
print(f"계정별 안전 한도까지 남은 호출: {MAX_API_CALLS_PER_ACCOUNT - api_calls_recorded:,}회")

## 3. Directions 5 호출·응답 처리 함수


In [ ]:
def haversine_m(lon1: float, lat1: float, lon2: float, lat2: float) -> float:
    lon1_rad, lat1_rad, lon2_rad, lat2_rad = map(
        math.radians, [lon1, lat1, lon2, lat2]
    )
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2) ** 2
    )
    return 6371008.8 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


session = requests.Session()
session.headers.update(
    {
        "x-ncp-apigw-api-key-id": NAVER_MAPS_CLIENT_ID,
        "x-ncp-apigw-api-key": NAVER_MAPS_CLIENT_SECRET,
    }
)

last_request_started_at = 0.0


def wait_for_rate_limit() -> None:
    global last_request_started_at
    minimum_interval = 1.0 / REQUESTS_PER_SECOND
    elapsed = time.monotonic() - last_request_started_at
    if elapsed < minimum_interval:
        time.sleep(minimum_interval - elapsed)
    last_request_started_at = time.monotonic()


def empty_result(row: pd.Series) -> dict:
    return {
        "route_request_id": row["route_request_id"],
        "request_batch_number": int(row["request_batch_number"]),
        "request_batch_id": row["request_batch_id"],
        "collector_label": TEAM_MEMBER_LABEL,
        "source_request_file_sha256": source_hash,
        "route_option": ROUTE_OPTION,
        "route_status": "API_ERROR",
        "route_check": "REVIEW",
        "review_flag": "",
        "http_status": np.nan,
        "api_code": np.nan,
        "api_message": "",
        "road_distance_m": np.nan,
        "road_duration_ms": np.nan,
        "road_duration_sec": np.nan,
        "road_duration_min": np.nan,
        "road_to_straight_ratio": np.nan,
        "api_current_datetime": "",
        "api_departure_time": "",
        "requested_at_kst": "",
        "collected_at_utc": "",
        "snapped_start_lon": np.nan,
        "snapped_start_lat": np.nan,
        "snapped_goal_lon": np.nan,
        "snapped_goal_lat": np.nan,
        "start_snap_distance_m": np.nan,
        "goal_snap_distance_m": np.nan,
        "ferry_flag": False,
        "attempt_count": 0,
        "last_error": "",
    }


def finish_error(result: dict, review_flag: str, message: str) -> dict:
    """개별 요청 오류를 결과로 확정한다. 예외를 다시 발생시키지 않는다."""
    result.update(
        {
            "route_status": "API_ERROR",
            "route_check": "REVIEW",
            "review_flag": review_flag,
            "last_error": message,
            "collected_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
    )
    return result


def collect_one_route(row: pd.Series) -> dict:
    result = empty_result(row)

    try:
        start_lon = float(row["origin_lon"])
        start_lat = float(row["origin_lat"])
        goal_lon = float(row["destination_school_lon"])
        goal_lat = float(row["destination_school_lat"])
        straight_distance_km = float(row["straight_distance_km"])
    except Exception as error:
        return finish_error(
            result,
            "INVALID_INPUT_COORDINATES",
            f"{type(error).__name__}: {error}",
        )

    params = {
        "start": f"{start_lon:.8f},{start_lat:.8f}",
        "goal": f"{goal_lon:.8f},{goal_lat:.8f}",
        "option": ROUTE_OPTION,
        "lang": "ko",
    }

    for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
        result["attempt_count"] = attempt
        result["requested_at_kst"] = datetime.now(KST).isoformat(timespec="seconds")
        wait_for_rate_limit()

        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=(CONNECT_TIMEOUT_SEC, READ_TIMEOUT_SEC),
            )
            result["http_status"] = response.status_code
            body_excerpt = " ".join((response.text or "").split())[:500]

            try:
                payload = response.json()
                if not isinstance(payload, dict):
                    payload = {}
            except ValueError:
                payload = {}

            api_code = payload.get("code", "")
            api_message = payload.get("message", "") or body_excerpt
            result["api_code"] = api_code
            result["api_message"] = api_message
            result["api_current_datetime"] = payload.get("currentDateTime", "")

            # 성공
            if response.status_code == 200 and str(api_code) == "0":
                route_list = payload.get("route", {}).get(ROUTE_OPTION, [])
                if not route_list:
                    raise ValueError(f"응답에 route.{ROUTE_OPTION}이 없습니다.")
                route = route_list[0]
                summary = route["summary"]
                road_distance_m = float(summary["distance"])
                road_duration_ms = float(summary["duration"])
                snapped_start = summary["start"]["location"]
                snapped_goal = summary["goal"]["location"]

                start_snap_m = haversine_m(
                    start_lon, start_lat, float(snapped_start[0]), float(snapped_start[1])
                )
                goal_snap_m = haversine_m(
                    goal_lon, goal_lat, float(snapped_goal[0]), float(snapped_goal[1])
                )
                ferry_flag = any(
                    int(guide.get("type", -1)) in {48, 49}
                    for guide in route.get("guide", [])
                )
                ratio = (
                    road_distance_m / (straight_distance_km * 1000)
                    if straight_distance_km > 0
                    else np.nan
                )

                review_reasons = []
                if ferry_flag:
                    review_reasons.append("FERRY_ROUTE")
                if start_snap_m > 500:
                    review_reasons.append("START_SNAP_OVER_500M")
                if goal_snap_m > 500:
                    review_reasons.append("GOAL_SNAP_OVER_500M")
                if pd.notna(ratio) and ratio > 5:
                    review_reasons.append("ROAD_TO_STRAIGHT_OVER_5")
                if road_duration_ms / 60000 > 180:
                    review_reasons.append("ROAD_DURATION_OVER_180MIN")

                result.update(
                    {
                        "route_status": "SUCCESS",
                        "route_check": "REVIEW" if review_reasons else "PASS",
                        "review_flag": "|".join(review_reasons),
                        "road_distance_m": road_distance_m,
                        "road_duration_ms": road_duration_ms,
                        "road_duration_sec": road_duration_ms / 1000,
                        "road_duration_min": road_duration_ms / 60000,
                        "road_to_straight_ratio": ratio,
                        "api_departure_time": summary.get("departureTime", ""),
                        "snapped_start_lon": float(snapped_start[0]),
                        "snapped_start_lat": float(snapped_start[1]),
                        "snapped_goal_lon": float(snapped_goal[0]),
                        "snapped_goal_lat": float(snapped_goal[1]),
                        "start_snap_distance_m": start_snap_m,
                        "goal_snap_distance_m": goal_snap_m,
                        "ferry_flag": ferry_flag,
                        "last_error": "",
                    }
                )
                result["collected_at_utc"] = datetime.now(timezone.utc).isoformat(
                    timespec="seconds"
                )
                return result

            # 출발·도착 동일, 도로 주변 아님, 자동차 경로 없음 등 공식 경로 없음
            if response.status_code == 400 and str(api_code) in {"1", "2", "3", "4", "5"}:
                result.update(
                    {
                        "route_status": "NO_ROUTE",
                        "route_check": "REVIEW",
                        "review_flag": f"DIRECTIONS_CODE_{api_code}",
                        "last_error": api_message or f"Directions code {api_code}",
                        "collected_at_utc": datetime.now(timezone.utc).isoformat(
                            timespec="seconds"
                        ),
                    }
                )
                return result

            # 인증 오류는 재시도의 의미가 없지만 해당 요청만 오류로 기록하고 계속한다.
            if response.status_code in {401, 403}:
                return finish_error(
                    result,
                    f"AUTH_OR_PERMISSION_HTTP_{response.status_code}",
                    f"HTTP {response.status_code}, code={api_code}, message={api_message}",
                )

            # 그 밖의 HTTP 오류는 일시 오류일 수 있으므로 제한 횟수만큼 재시도한다.
            result["last_error"] = (
                f"HTTP {response.status_code}, code={api_code}, message={api_message}"
            )
            if attempt < MAX_RETRIES_PER_REQUEST:
                if response.status_code == 429 or response.status_code >= 500:
                    time.sleep(min(60, 2 ** attempt))
                else:
                    time.sleep(min(10, 2 ** attempt))
                continue

            code_suffix = f"_CODE_{api_code}" if str(api_code) else "_EMPTY_CODE"
            return finish_error(
                result,
                f"HTTP_{response.status_code}{code_suffix}",
                result["last_error"],
            )

        except (requests.Timeout, requests.ConnectionError) as error:
            result["last_error"] = f"{type(error).__name__}: {error}"
            if attempt < MAX_RETRIES_PER_REQUEST:
                time.sleep(min(60, 2 ** attempt))
                continue
            return finish_error(result, "NETWORK_ERROR", result["last_error"])
        except Exception as error:
            result["last_error"] = f"{type(error).__name__}: {error}"
            if attempt < MAX_RETRIES_PER_REQUEST:
                time.sleep(min(10, 2 ** attempt))
                continue
            return finish_error(result, "RESPONSE_PARSE_ERROR", result["last_error"])

    return finish_error(result, "UNEXPECTED_RETRY_EXIT", result["last_error"])


def append_checkpoint(rows: list[dict]) -> None:
    if not rows:
        return
    checkpoint = pd.DataFrame(rows, columns=RESULT_COLUMNS)
    write_header = not RESULT_PATH.exists() or RESULT_PATH.stat().st_size == 0
    checkpoint.to_csv(
        RESULT_PATH,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig" if write_header else "utf-8",
        float_format="%.10f",
    )


def latest_result_per_request(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        return raw.copy()
    work = raw.copy()
    work["_row_order"] = np.arange(len(work))
    work["_status_priority"] = work["route_status"].map(
        {"SUCCESS": 4, "NO_ROUTE": 3, "API_ERROR": 2, "RETRYABLE_ERROR": 1}
    ).fillna(0)
    work["_collected_at"] = pd.to_datetime(work["collected_at_utc"], errors="coerce", utc=True)
    work = work.sort_values(
        ["route_request_id", "_status_priority", "_collected_at", "_row_order"],
        kind="stable",
    )
    return work.drop_duplicates("route_request_id", keep="last").drop(
        columns=["_row_order", "_status_priority", "_collected_at"]
    )


def load_fresh_resume_state() -> tuple[pd.DataFrame, int, set[str]]:
    """Drive의 최신 체크포인트를 매 실행 직전에 다시 읽는다."""
    if RESULT_PATH.exists() and RESULT_PATH.stat().st_size > 0:
        raw = read_csv_text(RESULT_PATH)
    else:
        raw = pd.DataFrame(columns=RESULT_COLUMNS)

    missing = [column for column in RESULT_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"체크포인트 필수 칼럼 누락: {missing}")

    unexpected_ids = set(raw["route_request_id"].dropna()) - set(
        batch_requests["route_request_id"]
    )
    if unexpected_ids:
        raise ValueError(
            "체크포인트에 다른 배치 ID가 포함되어 있습니다: "
            f"{list(sorted(unexpected_ids))[:5]}"
        )

    terminal_rows = raw[raw["route_status"].isin(TERMINAL_STATUSES)]
    fresh_done_ids = set(terminal_rows["route_request_id"])
    fresh_pending = batch_requests[
        ~batch_requests["route_request_id"].isin(fresh_done_ids)
    ].copy()
    recorded_calls = int(
        pd.to_numeric(raw.get("attempt_count"), errors="coerce").fillna(0).sum()
    )
    return fresh_pending, recorded_calls, fresh_done_ids


def write_collection_summary(stop_reason: str = "") -> pd.DataFrame:
    if RESULT_PATH.exists() and RESULT_PATH.stat().st_size > 0:
        raw = read_csv_text(RESULT_PATH)
    else:
        raw = pd.DataFrame(columns=RESULT_COLUMNS)
    latest = latest_result_per_request(raw)
    latest = latest[latest["route_request_id"].isin(set(batch_requests["route_request_id"]))]
    status_counts = latest["route_status"].value_counts().to_dict()
    call_count = int(pd.to_numeric(raw.get("attempt_count"), errors="coerce").fillna(0).sum())
    completed = int(latest["route_status"].isin(TERMINAL_STATUSES).sum())
    remaining = len(batch_requests) - completed
    api_errors = int(status_counts.get("API_ERROR", 0))

    if remaining == 0:
        run_status = "COMPLETE_WITH_ERRORS" if api_errors else "COMPLETE"
    else:
        run_status = "STOPPED" if stop_reason else "PARTIAL"

    summary = pd.DataFrame(
        [
            {"metric": "run_status", "value": run_status, "note": stop_reason},
            {"metric": "collector_label", "value": TEAM_MEMBER_LABEL, "note": ""},
            {"metric": "request_batch_id", "value": BATCH_ID, "note": ""},
            {"metric": "source_request_file_sha256", "value": source_hash, "note": ""},
            {"metric": "assigned_route_count", "value": len(batch_requests), "note": ""},
            {"metric": "raw_checkpoint_row_count", "value": len(raw), "note": "재시도·이전 오류 행 포함"},
            {"metric": "unique_route_result_count", "value": len(latest), "note": ""},
            {"metric": "success_count", "value": status_counts.get("SUCCESS", 0), "note": ""},
            {"metric": "no_route_count", "value": status_counts.get("NO_ROUTE", 0), "note": ""},
            {"metric": "api_error_count", "value": api_errors, "note": "오류 기록 후 수집 계속"},
            {"metric": "retryable_error_count", "value": status_counts.get("RETRYABLE_ERROR", 0), "note": "구버전 오류는 재실행 대상"},
            {"metric": "terminal_completed_count", "value": completed, "note": "SUCCESS+NO_ROUTE+API_ERROR"},
            {"metric": "remaining_route_count", "value": remaining, "note": "재실행 시 최신 체크포인트 기준 자동 재개"},
            {"metric": "recorded_api_call_attempts", "value": call_count, "note": "체크포인트에 기록된 최소 호출 수"},
            {"metric": "max_api_calls_per_account", "value": MAX_API_CALLS_PER_ACCOUNT, "note": "비용·할당량 보호용 중단선"},
            {"metric": "requests_per_second", "value": REQUESTS_PER_SECOND, "note": ""},
            {"metric": "route_option", "value": ROUTE_OPTION, "note": "실시간 최적"},
            {"metric": "result_file", "value": RESULT_PATH.name, "note": "팀장이 받을 핵심 파일"},
        ]
    )
    summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
    return summary

## 4. 본 수집 실행

중간에 Colab이 끊기면 이 셀만 다시 실행해도 됩니다.
이 셀은 실행할 때마다 Drive의 최신 결과 파일을 다시 읽습니다.

- `SUCCESS`, `NO_ROUTE`, `API_ERROR`가 기록된 요청은 자동으로 건너뜁니다.
- 구버전에서 남은 `RETRYABLE_ERROR`는 다시 시도합니다.
- 진행률은 남은 작업을 0%부터 표시하지 않고 전체 배치의 실제 완료 지점부터 표시합니다.
- 개별 요청 오류는 결과에 기록하고 다음 요청을 계속 처리합니다.

In [ ]:
# 중요: 이 셀을 다시 실행할 때마다 Drive의 최신 체크포인트를 다시 읽는다.
pending_requests, api_calls_recorded, done_ids = load_fresh_resume_state()
completed_before_run = len(done_ids)

print(f"전체 담당 경로: {len(batch_requests):,}건")
print(f"기존 완료·오류 기록으로 건너뛸 경로: {completed_before_run:,}건")
print(f"이번 실행에서 실제 호출할 경로: {len(pending_requests):,}건")
print(f"체크포인트 기록 호출 수: {api_calls_recorded:,}회")
if not pending_requests.empty:
    print(f"재개 첫 요청 ID: {pending_requests.iloc[0]['route_request_id']}")

buffer = []
api_calls_this_run = 0
stop_reason = ""

try:
    progress = tqdm(
        pending_requests.iterrows(),
        total=len(batch_requests),
        initial=completed_before_run,
        desc=f"Collecting {BATCH_ID}",
    )
    for _, request_row in progress:
        # 비용·할당량 보호를 위한 유일한 자동 중단 조건
        if (
            api_calls_recorded + api_calls_this_run + MAX_RETRIES_PER_REQUEST
            > MAX_API_CALLS_PER_ACCOUNT
        ):
            stop_reason = (
                f"비용·할당량 보호 한도 {MAX_API_CALLS_PER_ACCOUNT:,}회에 도달하여 중단"
            )
            break

        try:
            collected = collect_one_route(request_row)
        except Exception as error:
            # collect_one_route 내부에서 잡지 못한 예외도 해당 요청만 기록한다.
            collected = empty_result(request_row)
            collected = finish_error(
                collected,
                "UNHANDLED_REQUEST_EXCEPTION",
                f"{type(error).__name__}: {error}",
            )

        api_calls_this_run += int(collected.get("attempt_count", 0))
        buffer.append(collected)

        if len(buffer) >= CHECKPOINT_EVERY:
            append_checkpoint(buffer)
            buffer.clear()

        progress.set_postfix(
            status=collected["route_status"],
            calls=api_calls_recorded + api_calls_this_run,
        )

except KeyboardInterrupt:
    stop_reason = "사용자 중단 또는 Colab 실행 중단"
finally:
    append_checkpoint(buffer)
    buffer.clear()
    collection_summary = write_collection_summary(stop_reason)

display(collection_summary)
print(f"결과 저장: {RESULT_PATH}")
print(f"요약 저장: {SUMMARY_PATH}")
if stop_reason:
    print(f"중단 사유: {stop_reason}")
else:
    print("수집 셀이 정상 종료되었습니다. 개별 오류는 API_ERROR로 기록되었습니다.")

## 5. 팀장에게 전달할 파일 내려받기

요약표의 `run_status`가 `COMPLETE` 또는 `COMPLETE_WITH_ERRORS`인지 확인한 뒤
아래 셀을 실행합니다. 핵심 결과 CSV와 요약 CSV 두 개를 팀장에게 전달하세요.

`COMPLETE_WITH_ERRORS`라면 전체 요청을 모두 처리했지만 일부 요청이
`API_ERROR`로 기록되었다는 뜻입니다. 팀장이 병합·검증 단계에서 오류 행을 분리합니다.

In [ ]:
from google.colab import files

files.download(str(RESULT_PATH))
files.download(str(SUMMARY_PATH))
